# SWAN-SF Train/Test Standardization - Memory-Efficient Version

This notebook creates model-ready train/test arrays for the SWAN-SF tensor partitions without concatenating all partitions into RAM.

**Split:**

- Train = partitions **1, 2, 3, 5**
- Test = partition **4**

**Main idea:**

1. Load one partition at a time.
2. Compute feature-wise train mean/std using only train partitions.
3. Re-load one partition at a time.
4. Standardize in chunks.
5. Save directly to `.npy` files in **aeon format**: `(n_cases, n_channels, n_timepoints)`.

For your cleaned tensors, the expected shape is usually:

```text
(N, 60, 47)   # original tensor format
(N, 47, 60)   # aeon format after transpose
```

The notebook avoids creating a full-size `X_train_std` copy in memory.

## 1. Imports and paths

Edit `BASE_DIR` and file names so they match your Google Drive folder.

In [ ]:
import gc
import json
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

In [ ]:
# Optional: mount Google Drive in Colab.
# Run this cell if your files are stored in Google Drive.
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
# -----------------------------
# User configuration
# -----------------------------

# CHANGE THIS to the folder where your partition .npz files are stored.
BASE_DIR = Path("/content/drive/MyDrive/solar_flare_forecasting")


DATA_DIR = Path('/content/drive/MyDrive/solar_flare_forecasting/standardized_tensors')

partition_paths = {
    1: DATA_DIR / 'partition1_combined_clean.npz',
    2: DATA_DIR / 'partition2_combined_clean.npz',
    3: DATA_DIR / 'partition3_combined_clean.npz',
    4: DATA_DIR / 'partition4_combined_clean.npz',
    5: DATA_DIR / 'partition5_combined_clean.npz',
}

TRAIN_PARTITIONS = [1, 2, 3, 5]
TEST_PARTITIONS = [4]

# Based on your inspected file:
X_KEY = "X"
Y_KEY = "y_flare"

# Chunk size controls RAM usage during standardization and validation.
# If Colab still runs out of RAM, lower this to 2048 or 1024.
CHUNK_SIZE = 4096

# Output folder in Google Drive.
OUT_DIR = BASE_DIR / "model_ready_partition_split"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUT_DIR)

## 2. Helper functions

In [ ]:
def memory_gb_from_shape(shape, dtype=np.float32):
    # Return estimated memory in GB for an array shape and dtype.
    return np.prod(shape) * np.dtype(dtype).itemsize / (1024 ** 3)


def class_count_dict(y):
    # Return class counts with string keys so the result is JSON-friendly.
    values, counts = np.unique(y, return_counts=True)
    return {str(v): int(c) for v, c in zip(values, counts)}


def inspect_npz(path):
    # Print the keys, shapes, and dtypes inside an .npz file.
    path = Path(path)
    print(f"File: {path.name}")
    with np.load(path, allow_pickle=True) as data:
        print("Keys:", list(data.files))
        for key in data.files:
            arr = data[key]
            print(f"  {key}: shape={arr.shape}, dtype={arr.dtype}")


def summarize_partition(path, x_key=X_KEY, y_key=Y_KEY):
    # Load one partition and return basic validation information.
    path = Path(path)
    with np.load(path, allow_pickle=True) as data:
        X = data[x_key]
        y = data[y_key]

        summary = {
            "file": path.name,
            "X_shape": tuple(int(v) for v in X.shape),
            "y_shape": tuple(int(v) for v in y.shape),
            "X_dtype": str(X.dtype),
            "y_dtype": str(y.dtype),
            "X_memory_GB": round(memory_gb_from_shape(X.shape, X.dtype), 3),
            "NaN_count": int(np.isnan(X).sum()),
            "Inf_count": int(np.isinf(X).sum()),
            "class_counts": class_count_dict(y),
        }

    gc.collect()
    return summary


def update_running_feature_sums(X, feature_sum, feature_sumsq, total_observations, chunk_size=CHUNK_SIZE):
    # Update running feature sums and squared sums.
    # X is expected in original format: (n_cases, n_timepoints, n_features).
    n_cases = X.shape[0]

    for start in range(0, n_cases, chunk_size):
        end = min(start + chunk_size, n_cases)
        chunk = X[start:end]

        # Sum over cases and timesteps, leaving one value per feature.
        feature_sum += chunk.sum(axis=(0, 1), dtype=np.float64)

        # Squaring in float64 improves numerical stability without converting the whole partition.
        feature_sumsq += np.square(chunk, dtype=np.float64).sum(axis=(0, 1))

        total_observations += chunk.shape[0] * chunk.shape[1]

    return feature_sum, feature_sumsq, total_observations


def validate_saved_aeon_array(path, chunk_size=CHUNK_SIZE):
    # Validate a saved .npy array in aeon format: (n_cases, n_channels, n_timepoints).
    X = np.load(path, mmap_mode="r")
    n_cases, n_channels, n_timepoints = X.shape

    nan_count = 0
    inf_count = 0
    feature_sum = np.zeros(n_channels, dtype=np.float64)
    feature_sumsq = np.zeros(n_channels, dtype=np.float64)
    total_observations = 0

    for start in tqdm(range(0, n_cases, chunk_size), desc=f"Validating {Path(path).name}"):
        end = min(start + chunk_size, n_cases)
        chunk = X[start:end]

        nan_count += int(np.isnan(chunk).sum())
        inf_count += int(np.isinf(chunk).sum())

        # aeon format is (cases, channels, timepoints), so sum over cases and timepoints.
        feature_sum += chunk.sum(axis=(0, 2), dtype=np.float64)
        feature_sumsq += np.square(chunk, dtype=np.float64).sum(axis=(0, 2))
        total_observations += chunk.shape[0] * chunk.shape[2]

    mean = feature_sum / total_observations
    variance = np.maximum((feature_sumsq / total_observations) - (mean ** 2), 0.0)
    std = np.sqrt(variance)

    return {
        "shape": tuple(int(v) for v in X.shape),
        "dtype": str(X.dtype),
        "memory_GB": round(memory_gb_from_shape(X.shape, X.dtype), 3),
        "NaN_count": int(nan_count),
        "Inf_count": int(inf_count),
        "feature_mean_min": float(mean.min()),
        "feature_mean_max": float(mean.max()),
        "feature_std_min": float(std.min()),
        "feature_std_max": float(std.max()),
    }

## 3. Inspect partition keys

This confirms the correct keys before doing the full processing.

For your current files, you should see:

```python
X_KEY = "X"
Y_KEY = "y_flare"
```

In [ ]:
inspect_npz(partition_paths[1])

## 4. Validate raw partitions one at a time

This replaces the old approach of loading all train partitions into `X_train_list` and concatenating them.

The table should show:

- `NaN_count = 0`
- `Inf_count = 0`
- matching first dimensions between `X_shape` and `y_shape`
- consistent `(60, n_features)` across all partitions

In [ ]:
partition_summaries = []

for p in tqdm(TRAIN_PARTITIONS + TEST_PARTITIONS, desc="Summarizing raw partitions"):
    summary = summarize_partition(partition_paths[p])
    summary["partition"] = p
    summary["split"] = "train" if p in TRAIN_PARTITIONS else "test"
    partition_summaries.append(summary)

summary_df = pd.DataFrame(partition_summaries)
summary_df

In [ ]:
# Validate shape consistency and compute total train/test sample counts.

def get_shape_from_summary(partition):
    row = summary_df.loc[summary_df["partition"] == partition].iloc[0]
    return tuple(row["X_shape"])

reference_shape = get_shape_from_summary(TRAIN_PARTITIONS[0])
n_timepoints = reference_shape[1]
n_features = reference_shape[2]

for row in partition_summaries:
    X_shape = row["X_shape"]
    y_shape = row["y_shape"]
    assert X_shape[0] == y_shape[0], f"Sample mismatch in partition {row['partition']}"
    assert X_shape[1] == n_timepoints, f"Timepoint mismatch in partition {row['partition']}"
    assert X_shape[2] == n_features, f"Feature mismatch in partition {row['partition']}"
    assert row["NaN_count"] == 0, f"NaNs found in partition {row['partition']}"
    assert row["Inf_count"] == 0, f"Infs found in partition {row['partition']}"

n_train = int(summary_df.loc[summary_df["split"] == "train", "X_shape"].apply(lambda s: s[0]).sum())
n_test = int(summary_df.loc[summary_df["split"] == "test", "X_shape"].apply(lambda s: s[0]).sum())

print("n_train:", n_train)
print("n_test:", n_test)
print("n_timepoints:", n_timepoints)
print("n_features:", n_features)
print("Expected original format: (N, 60, 47) if using your cleaned tensors")
print("Expected aeon format:     (N, 47, 60) if using your cleaned tensors")

In [ ]:
# Combined class counts without concatenating y arrays.
combined_class_counts = {
    "train": {},
    "test": {},
}

for row in partition_summaries:
    split = row["split"]
    for label, count in row["class_counts"].items():
        combined_class_counts[split][label] = combined_class_counts[split].get(label, 0) + int(count)

print("Combined class counts")
print(json.dumps(combined_class_counts, indent=2))

## 5. Fit feature-wise scaler on train only

This computes:

```text
mean_j = mean of X_train[:, :, j]
std_j  = std  of X_train[:, :, j]
```

using only partitions **1, 2, 3, and 5**.

Partition 4 is not used here, because using test data to fit the scaler would cause data leakage.

In [ ]:
feature_sum = np.zeros(n_features, dtype=np.float64)
feature_sumsq = np.zeros(n_features, dtype=np.float64)
total_observations = 0

for p in tqdm(TRAIN_PARTITIONS, desc="First pass: computing train scaler"):
    with np.load(partition_paths[p], allow_pickle=True) as data:
        X = data[X_KEY]
        feature_sum, feature_sumsq, total_observations = update_running_feature_sums(
            X,
            feature_sum,
            feature_sumsq,
            total_observations,
            chunk_size=CHUNK_SIZE,
        )
    del X
    gc.collect()

feature_mean_1d = feature_sum / total_observations
feature_var_1d = np.maximum((feature_sumsq / total_observations) - (feature_mean_1d ** 2), 0.0)
feature_std_1d = np.sqrt(feature_var_1d)

# Protect against divide-by-zero if a feature is constant.
epsilon = 1e-8
zero_std_mask = feature_std_1d < epsilon
feature_std_safe_1d = feature_std_1d.copy()
feature_std_safe_1d[zero_std_mask] = 1.0

# Broadcastable shapes for original tensor format: (N, timepoints, features).
feature_mean = feature_mean_1d.astype(np.float32).reshape(1, 1, n_features)
feature_std = feature_std_1d.astype(np.float32).reshape(1, 1, n_features)
feature_std_safe = feature_std_safe_1d.astype(np.float32).reshape(1, 1, n_features)

print("feature_mean shape:", feature_mean.shape)
print("feature_std shape:", feature_std.shape)
print("zero-std feature count:", int(zero_std_mask.sum()))
print("mean range:", float(feature_mean_1d.min()), "to", float(feature_mean_1d.max()))
print("std range:", float(feature_std_1d.min()), "to", float(feature_std_1d.max()))

In [ ]:
# Save scaler parameters immediately.
scaler_path = OUT_DIR / "scaler_params.npz"

np.savez(
    scaler_path,
    feature_mean=feature_mean,
    feature_std=feature_std,
    feature_std_safe=feature_std_safe,
    feature_mean_1d=feature_mean_1d.astype(np.float32),
    feature_std_1d=feature_std_1d.astype(np.float32),
    zero_std_mask=zero_std_mask,
    train_partitions=np.array(TRAIN_PARTITIONS, dtype=np.int16),
    test_partitions=np.array(TEST_PARTITIONS, dtype=np.int16),
)

print("Saved scaler parameters to:", scaler_path)

## 6. Standardize in chunks and save directly to `.npy`

This is the memory-safe replacement for:

```python
X_train_std = (X_train - feature_mean) / feature_std
```

Instead of creating a huge standardized copy in RAM, this section writes directly to disk using NumPy memory-mapped `.npy` files.

Output format is **aeon-ready**:

```text
(n_cases, n_channels, n_timepoints)
```

For your cleaned tensors, that should be:

```text
Train: (277010, 47, 60)
Test:  (50963, 47, 60)
```

In [ ]:
X_train_out_path = OUT_DIR / "X_train_standardized_aeon.npy"
y_train_out_path = OUT_DIR / "y_train.npy"
X_test_out_path = OUT_DIR / "X_test_standardized_aeon.npy"
y_test_out_path = OUT_DIR / "y_test.npy"

print("Train X output:", X_train_out_path)
print("Train y output:", y_train_out_path)
print("Test X output:", X_test_out_path)
print("Test y output:", y_test_out_path)

In [ ]:
def standardize_partitions_to_mmap(partitions, X_out_path, y_out_path, total_cases, split_name):
    # Load partitions one at a time, standardize in chunks, transpose to aeon format,
    # and write directly to .npy files.
    X_out = np.lib.format.open_memmap(
        X_out_path,
        mode="w+",
        dtype=np.float32,
        shape=(total_cases, n_features, n_timepoints),
    )

    y_out = np.lib.format.open_memmap(
        y_out_path,
        mode="w+",
        dtype=np.int8,
        shape=(total_cases,),
    )

    write_index = 0

    for p in tqdm(partitions, desc=f"Second pass: writing {split_name}"):
        with np.load(partition_paths[p], allow_pickle=True) as data:
            X = data[X_KEY]
            y = data[Y_KEY]

            n_cases_partition = X.shape[0]

            for start in range(0, n_cases_partition, CHUNK_SIZE):
                end = min(start + CHUNK_SIZE, n_cases_partition)
                chunk = X[start:end]
                y_chunk = y[start:end]

                # Standardize in original format: (batch, timepoints, features).
                chunk_std = (chunk - feature_mean) / feature_std_safe

                # Convert to aeon format: (batch, features/channels, timepoints).
                chunk_aeon = np.transpose(chunk_std, (0, 2, 1)).astype(np.float32, copy=False)

                batch_size = end - start
                X_out[write_index:write_index + batch_size] = chunk_aeon
                y_out[write_index:write_index + batch_size] = y_chunk.astype(np.int8, copy=False)

                write_index += batch_size

        del X, y
        gc.collect()

    assert write_index == total_cases, f"Expected {total_cases} cases, wrote {write_index}"

    X_out.flush()
    y_out.flush()

    del X_out, y_out
    gc.collect()


standardize_partitions_to_mmap(
    TRAIN_PARTITIONS,
    X_train_out_path,
    y_train_out_path,
    n_train,
    split_name="train",
)

standardize_partitions_to_mmap(
    TEST_PARTITIONS,
    X_test_out_path,
    y_test_out_path,
    n_test,
    split_name="test",
)

print("Finished writing standardized aeon-format arrays.")

## 7. Validate saved standardized data

The train standardized means should be very close to 0 and train standardized stds should be very close to 1.

The test set does **not** need to have mean 0 or std 1, because the scaler was fit only on train data.

In [ ]:
train_saved_summary = validate_saved_aeon_array(X_train_out_path)
test_saved_summary = validate_saved_aeon_array(X_test_out_path)

print("TRAIN SAVED SUMMARY")
print(json.dumps(train_saved_summary, indent=2))

print("TEST SAVED SUMMARY")
print(json.dumps(test_saved_summary, indent=2))

In [ ]:
# Validate label arrays.
y_train_saved = np.load(y_train_out_path, mmap_mode="r")
y_test_saved = np.load(y_test_out_path, mmap_mode="r")

label_summary = {
    "y_train_shape": tuple(int(v) for v in y_train_saved.shape),
    "y_test_shape": tuple(int(v) for v in y_test_saved.shape),
    "y_train_dtype": str(y_train_saved.dtype),
    "y_test_dtype": str(y_test_saved.dtype),
    "y_train_class_counts": class_count_dict(y_train_saved),
    "y_test_class_counts": class_count_dict(y_test_saved),
}

print(json.dumps(label_summary, indent=2))

## 8. Save metadata summary

This JSON file is not required for model training, but it is useful for keeping track of exactly how the model-ready data was created.

In [ ]:
metadata = {
    "project": "SWAN-SF solar flare classification",
    "notebook_purpose": "Memory-efficient train/test split and train-only standardization",
    "train_partitions": TRAIN_PARTITIONS,
    "test_partitions": TEST_PARTITIONS,
    "x_key": X_KEY,
    "y_key": Y_KEY,
    "chunk_size": CHUNK_SIZE,
    "original_tensor_format": "(n_cases, n_timepoints, n_features)",
    "aeon_tensor_format": "(n_cases, n_channels, n_timepoints)",
    "n_train": n_train,
    "n_test": n_test,
    "n_timepoints": n_timepoints,
    "n_features": n_features,
    "standardization": {
        "method": "feature-wise z-score",
        "fit_on": "train partitions only",
        "mean_axes_original_format": "axis=(0, 1), over cases and timesteps",
        "std_axes_original_format": "axis=(0, 1), over cases and timesteps",
        "zero_std_feature_count": int(zero_std_mask.sum()),
    },
    "raw_partition_summaries": partition_summaries,
    "combined_class_counts": combined_class_counts,
    "saved_X_train_summary": train_saved_summary,
    "saved_X_test_summary": test_saved_summary,
    "saved_label_summary": label_summary,
    "output_files": {
        "X_train_standardized_aeon": str(X_train_out_path),
        "y_train": str(y_train_out_path),
        "X_test_standardized_aeon": str(X_test_out_path),
        "y_test": str(y_test_out_path),
        "scaler_params": str(scaler_path),
    },
    "important_note": "No train partitions are concatenated into one full X_train array in RAM. Standardized data is written directly to .npy files in chunks.",
}

metadata_path = OUT_DIR / "train_test_standardization_metadata.json"

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2, default=str)

print("Saved metadata to:", metadata_path)

## 9. Loading the saved arrays later

Use this in your model-training notebook.

In [ ]:
# Example loading code for a future notebook.
X_train = np.load(OUT_DIR / "X_train_standardized_aeon.npy", mmap_mode="r")
y_train = np.load(OUT_DIR / "y_train.npy", mmap_mode="r")
X_test = np.load(OUT_DIR / "X_test_standardized_aeon.npy", mmap_mode="r")
y_test = np.load(OUT_DIR / "y_test.npy", mmap_mode="r")

print("X_train:", X_train.shape, X_train.dtype)
print("y_train:", y_train.shape, y_train.dtype)
print("X_test:", X_test.shape, X_test.dtype)
print("y_test:", y_test.shape, y_test.dtype)

## 10. RAM cleanup

In [ ]:
del y_train_saved, y_test_saved
try:
    del X_train, X_test, y_train, y_test
except NameError:
    pass

gc.collect()
print("Cleanup complete.")